In [1]:
import sys
sys.path.append('../src')

from sklearn.model_selection import train_test_split
import numpy as np

# Bước 1 — Kiểm tra 2 file predictions đã khớp nhau

In [2]:
import pandas as pd

gb_pred = pd.read_csv("../outputs/metrics/gb_baseline_predictions.csv")
pretrained_pred = pd.read_csv("../outputs/metrics/pretrained_predictions.csv")

print("GB predictions:", gb_pred.shape, gb_pred.columns.tolist())
print("Pretrained predictions:", pretrained_pred.shape, pretrained_pred.columns.tolist())

# Kiểm tra 2 tập series có khớp nhau không (đề phòng lệch tập do sample khác nhau)
gb_ids = set(gb_pred["unique_id"])
pretrained_ids = set(pretrained_pred["unique_id"])
print("Series chỉ có ở GB:", len(gb_ids - pretrained_ids))
print("Series chỉ có ở Pretrained:", len(pretrained_ids - gb_ids))

GB predictions: (8334, 7) ['unique_id', 'length_group', 'category', 'ds', 'y', 'pred_lgb', 'pred_xgb']
Pretrained predictions: (8334, 6) ['unique_id', 'length_group', 'category', 'ds', 'y', 'pred_pretrained']
Series chỉ có ở GB: 0
Series chỉ có ở Pretrained: 0


# Bước 2 — Merge 2 nguồn dự báo theo (unique_id, ds)

In [3]:
merged = gb_pred.merge(
    pretrained_pred[["unique_id", "ds", "pred_pretrained"]],
    on=["unique_id", "ds"],
    how="inner"
)

assert merged["y_x"].equals(merged["y_y"]) if "y_x" in merged.columns else True
# Nếu cả 2 file đều có cột 'y' (giá trị thật), sau merge sẽ tạo y_x/y_y trùng lặp
# -> giữ 1 bản, xóa bản thừa
if "y_y" in merged.columns:
    merged = merged.drop(columns=["y_y"]).rename(columns={"y_x": "y"})

In [4]:
gb_pred = pd.read_csv("../outputs/metrics/gb_baseline_predictions.csv")
pretrained_pred = pd.read_csv("../outputs/metrics/pretrained_predictions.csv")

print("Cột trong gb_pred:", gb_pred.columns.tolist())
print("Cột trong pretrained_pred:", pretrained_pred.columns.tolist())

Cột trong gb_pred: ['unique_id', 'length_group', 'category', 'ds', 'y', 'pred_lgb', 'pred_xgb']
Cột trong pretrained_pred: ['unique_id', 'length_group', 'category', 'ds', 'y', 'pred_pretrained']


In [5]:
print(gb_pred.columns.tolist())

['unique_id', 'length_group', 'category', 'ds', 'y', 'pred_lgb', 'pred_xgb']


# Bước 3 — Xác định model GB nào đại diện

Theo thiết kế ban đầu: chọn model tốt hơn giữa LightGBM/XGBoost làm đại diện GB baseline cho phần ensemble (tránh làm loãng thí nghiệm với 2 model GB cùng lúc).

In [6]:
print("Khoảng ds trong gb_baseline_predictions.csv:")
print(gb_pred["ds"].min(), "->", gb_pred["ds"].max())

print("\nKhoảng ds trong pretrained_predictions.csv:")
print(pretrained_pred["ds"].min(), "->", pretrained_pred["ds"].max())

Khoảng ds trong gb_baseline_predictions.csv:
1902-02-01 -> 1932-07-01

Khoảng ds trong pretrained_predictions.csv:
1902-02-01 -> 1932-07-01


In [7]:
from evaluation import calculate_metrics

metrics_lgb = calculate_metrics(merged["y"], merged["pred_lgb"])
metrics_xgb = calculate_metrics(merged["y"], merged["pred_xgb"])

best_gb_col = "pred_lgb" if metrics_lgb["RMSE"] < metrics_xgb["RMSE"] else "pred_xgb"
print(f"Model GB đại diện: {best_gb_col}")

merged["pred_gb"] = merged[best_gb_col]

Model GB đại diện: pred_lgb


# Bước 4 - Tạo các phương án Ensemble (Simple Average & Weighted Average)

Tạo toàn bộ các cột dự báo ensemble trước khi tách tune/eval set,
để tránh lỗi thiếu cột khi eval_set được cắt ra quá sớm.

In [8]:
# Bước 4 & 5 gộp lại: tạo TẤT CẢ cột dự báo trước, rồi mới split

# Simple average
merged["pred_ensemble_avg"] = (merged["pred_gb"] + merged["pred_pretrained"]) / 2

# Split để tune weighted average (không leakage)
unique_ids = merged["unique_id"].unique()
tune_ids, eval_ids = train_test_split(unique_ids, test_size=0.3, random_state=42)

tune_set = merged[merged["unique_id"].isin(tune_ids)]

# Grid search trên tune_set
best_w, best_rmse = None, np.inf
for w in np.arange(0, 1.05, 0.05):
    pred = w * tune_set["pred_gb"] + (1 - w) * tune_set["pred_pretrained"]
    rmse = calculate_metrics(tune_set["y"], pred)["RMSE"]
    if rmse < best_rmse:
        best_rmse, best_w = rmse, w

print(f"Trọng số tối ưu: w={best_w:.2f}")

# Thêm cột weighted vào TOÀN BỘ merged trước
merged["pred_ensemble_weighted"] = (
    best_w * merged["pred_gb"] + (1 - best_w) * merged["pred_pretrained"]
)

# CUỐI CÙNG mới cắt eval_set — lúc này merged đã đủ mọi cột cần thiết
eval_set = merged[merged["unique_id"].isin(eval_ids)]

Trọng số tối ưu: w=0.80


# Bước 5 — So sánh 4 phương pháp trên eval_set (tập chưa dùng để tune)

In [9]:
methods = {
    "GB only": "pred_gb",
    "Pretrained only": "pred_pretrained",
    "Ensemble (avg)": "pred_ensemble_avg",
    "Ensemble (weighted)": "pred_ensemble_weighted",
}

print("Kết quả trên eval_set (không dùng để tune weight):")
for name, col in methods.items():
    m = calculate_metrics(eval_set["y"], eval_set[col])
    print(f"{name}: RMSE={m['RMSE']:.2f}, MAE={m['MAE']:.2f}")

Kết quả trên eval_set (không dùng để tune weight):
GB only: RMSE=896.24, MAE=417.41
Pretrained only: RMSE=1034.77, MAE=544.64
Ensemble (avg): RMSE=848.40, MAE=444.04
Ensemble (weighted): RMSE=847.19, MAE=415.46


# Bước 7 — Lưu kết quả đầy đủ, kèm length_group để dùng cho bước phân tích chính

In [10]:
final_results = merged[[
    "unique_id", "length_group", "category", "ds", "y",
    "pred_gb", "pred_pretrained", "pred_ensemble_avg", "pred_ensemble_weighted"
]].copy()

final_results.to_csv("../outputs/metrics/ensemble_predictions.csv", index=False)
print(f"Đã lưu {len(final_results)} dòng vào ensemble_predictions.csv")

Đã lưu 8334 dòng vào ensemble_predictions.csv
